# Kaggle vLLM API Setup (Modeled after kaggle-llm-api)

This notebook follows the structure of the upstream kaggle-llm-api workflow, adapted for this project.
It launches a persistent OpenAI-compatible API using the locked model and ngrok tunneling.

## Workflow

1. Validate environment and GPU
2. Install dependencies
3. Configure model and required environment variables
4. Start the server script
5. Wait for readiness and print endpoint markers
6. Run basic API tests
7. Keep the kernel alive

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

try:
    import torch
    cuda_available = bool(torch.cuda.is_available())
    gpu_name = torch.cuda.get_device_name(0) if cuda_available else "CPU"
    gpu_memory_gb = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if cuda_available else 0.0
except Exception:
    cuda_available = False
    gpu_name = "CPU"
    gpu_memory_gb = 0.0

print(json.dumps({
    "cwd": str(Path.cwd()),
    "cuda_available": cuda_available,
    "gpu_name": gpu_name,
    "gpu_memory_gb": gpu_memory_gb
}, indent=2))

In [ ]:
packages = [
    "vllm>=0.6.0",
    "huggingface_hub>=0.23.0",
    "requests>=2.31.0",
    "protobuf>=4.25.3,<6"
]

for pkg in packages:
    print(f"Installing {pkg}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", pkg])

print("Dependency installation complete.")

## Configuration

Required environment variables:
- `HF_TOKEN`
- `NGROK_AUTHTOKEN` and/or `NGROK_API_KEY`

In [ ]:
import json
import os

MODEL_ID = "Deign86/deped-math-qwen2.5-7b-deped-math-merged"

EMBEDDED_NGROK_AUTHTOKEN = "3BvTTulVe9BSgd4bG4JY4Jbigtu_JLRdEmXK36wuHyj4PWZA"

if not os.getenv("NGROK_AUTHTOKEN", "").strip() and EMBEDDED_NGROK_AUTHTOKEN.strip():
    os.environ["NGROK_AUTHTOKEN"] = EMBEDDED_NGROK_AUTHTOKEN.strip()
    print("Injected embedded NGROK_AUTHTOKEN fallback into environment.")

hf_token_present = bool(os.getenv("HF_TOKEN", "").strip())
ngrok_authtoken_present = bool(os.getenv("NGROK_AUTHTOKEN", "").strip())
ngrok_api_key_present = bool(os.getenv("NGROK_API_KEY", "").strip())
ngrok_token_present = bool(os.getenv("NGROK_TOKEN", "").strip())

print(json.dumps({
    "model_id": MODEL_ID,
    "hf_token_present": hf_token_present,
    "ngrok_authtoken_present": ngrok_authtoken_present,
    "ngrok_api_key_present": ngrok_api_key_present,
    "ngrok_token_present": ngrok_token_present,
    "embedded_ngrok_token_enabled": True
}, indent=2))

if not hf_token_present:
    print("HF_TOKEN is not set; proceeding with anonymous model download.")
if not (ngrok_authtoken_present or ngrok_api_key_present or ngrok_token_present):
    raise RuntimeError("Set NGROK_AUTHTOKEN (recommended) or NGROK_API_KEY / NGROK_TOKEN in Kaggle secrets or environment variables.")


## Start API Server

This launches `serve_kaggle_llm_api.py` in the background.

In [ ]:
import base64
import os
import subprocess
import sys
from pathlib import Path

script_candidates = [
    Path("serve_kaggle_llm_api.py"),
    Path("/kaggle/working/serve_kaggle_llm_api.py")
]
script_path = next((p for p in script_candidates if p.exists()), None)
if script_path is None:
    embedded_script_b64 = (
        "..."
    )
    script_path = Path("/kaggle/working/serve_kaggle_llm_api.py")
    script_path.write_text(base64.b64decode(embedded_script_b64).decode("utf-8"), encoding="utf-8")
    print(f"Wrote embedded serving script to {script_path}")

if "server_proc" in globals() and server_proc is not None and server_proc.poll() is None:
    print("Existing server process detected; terminating it before restart.")
    server_proc.terminate()
    try:
        server_proc.wait(timeout=15)
    except Exception:
        server_proc.kill()

runtime_env = os.environ.copy()
runtime_env.setdefault("PYTHONUNBUFFERED", "1")

server_proc = subprocess.Popen(
    [sys.executable, str(script_path)],
    cwd=str(script_path.parent),
    env=runtime_env
)

print(f"Server process started. PID={server_proc.pid}")


## Wait for Readiness and Capture Endpoints

In [ ]:
RUNTIME_STATUS_PATH = Path("/kaggle/working/api_runtime_status.json")
deadline = time.time() + 20 * 60
status_payload = None

while time.time() < deadline:
    if server_proc.poll() is not None:
        raise RuntimeError("Server process exited before readiness. Check notebook logs above for details.")

    if RUNTIME_STATUS_PATH.exists():
        try:
            payload = json.loads(RUNTIME_STATUS_PATH.read_text(encoding="utf-8"))
            if payload.get("success") and payload.get("ngrok_public_url"):
                status_payload = payload
                break
        except Exception:
            pass

    time.sleep(5)

if status_payload is None:
    raise TimeoutError("Timed out waiting for ngrok endpoint in api_runtime_status.json.")

PUBLIC_URL = str(status_payload.get("ngrok_public_url", "")).strip()
OPENAI_BASE_URL = str(status_payload.get("openai_base_url", f"{PUBLIC_URL}/v1")).strip()

print("SERVER_READY: true")
print(f"NGROK_PUBLIC_URL: {PUBLIC_URL}")
print(f"OPENAI_BASE_URL: {OPENAI_BASE_URL}")
print(json.dumps(status_payload, indent=2))

## Basic API Test

In [ ]:
import requests

local_models = requests.get("http://127.0.0.1:8000/v1/models", timeout=20)
print(f"Local /v1/models status: {local_models.status_code}")
print(local_models.text[:1000])

public_models = requests.get(f"{PUBLIC_URL}/v1/models", timeout=30)
print(f"Public /v1/models status: {public_models.status_code}")
print(public_models.text[:1000])

## Keep Server Running

Keep this cell active to keep the API alive.

In [ ]:
print("Keep-alive loop started.")
print(f"OPENAI_BASE_URL: {OPENAI_BASE_URL}")

try:
    while True:
        if server_proc.poll() is not None:
            raise RuntimeError("Server process exited while keep-alive loop was running.")
        time.sleep(60)
        print(f"Heartbeat: {time.strftime('%Y-%m-%d %H:%M:%S')}")
except KeyboardInterrupt:
    print("Stopping keep-alive loop by user request.")
    if server_proc.poll() is None:
        server_proc.terminate()